# Lesson 18: 암호화된 영수증으로 AI 에이전트 보호하기

## 실습 노트북

이 노트북에서는 네 가지 연습을 진행합니다:

1. 에이전트 도구 호출에 대한 **첫 번째 영수증 서명** 및 검증.
2. **영수증 변조** 후 검증 실패 확인.
3. **세 개의 영수증 체인 구축** 및 체인 무결성 확인.
4. Microsoft Agent Framework 도구 호출을 래핑하여 모든 작업에서 영수증이 생성되도록 함.

모든 암호화 기본 요소는 잘 관리되는 라이브러리(`pynacl`의 Ed25519, `jcs`의 RFC 8785 정규화 JSON, Python 표준 라이브러리의 SHA-256 `hashlib`)에서 가져옵니다. 영수증 로직 자체는 읽고 수정할 수 있는 순수 Python입니다.

셀을 순서대로 실행하세요. 각 섹션은 짧고 독립적입니다.


## Setup

두 가지 의존성을 설치하세요. 두 라이선스 모두 관대합니다(Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## Helper Utilities

이 두 유틸리티는 base64url 인코딩(패딩 없음)과 임의 객체의 SHA-256 해싱을 처리합니다. 이들은 나머지 노트북이 영수증 로직 자체에 집중할 수 있도록 돕습니다.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: 첫 번째 영수증에 서명하기

우리의 **Contoso Travel** 에이전트가 고객을 위해 시드니에서 로스앤젤레스로 가는 항공편을 방금 조회했다고 상상해 보세요. 우리는 이 도구 호출을 서명된 영수증으로 기록하여 미래의 감사자가 우리를 신뢰하지 않고도 이를 검증할 수 있도록 하려 합니다.

### Step 1.1: 서명 키 생성하기

운영 환경에서는 에이전트의 서명 키가 하드웨어 보안 모듈(HSM), Azure Key Vault 또는 유사한 보호된 저장소에 보관됩니다. 이 강의에서는 메모리에서 새 키를 생성합니다.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: 영수증 페이로드 구축

페이로드에는 영수증이 증명하고자 하는 모든 내용이 포함됩니다: 누가 행위했는지, 어떤 도구를 사용했는지, 어떤 인자를 사용했는지, 어떤 결과가 돌아왔는지, 어떤 정책 하에서인지, 그리고 언제인지. 인자와 결과를 인라인으로 포함하지 않고 해시하여 영수증이 민감한 내용을 노출하지 않도록 합니다.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: 영수증에 서명하고 조립하기

세 단계:

1. 두 구현이 동일한 논리적 영수증을 생성할 때 바이트가 동일하도록 JCS를 사용하여 페이로드를 정규화합니다.
2. 정규화된 바이트를 SHA-256으로 해시합니다.
3. Ed25519 개인 키로 해시를 서명합니다.

서명은 원래 페이로드에 첨부되어 최종 영수증을 생성합니다.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### Step 1.4: 영수증 검증

검증은 과정을 역전시키는 것입니다. 서명을 제거하고, 정규 해시를 다시 계산하며, 영수증에 있는 공개 키로 서명을 확인합니다.

이 검증을 수행하는 감사자는 우리에게서 영수증 이외에 아무 것도 필요하지 않습니다. 호출할 서비스도, 조회할 키 디렉터리도, 신뢰도 필요 없습니다.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

`Receipt is valid: True`가 보여야 합니다. 에이전트가 처음으로 암호화 서명된 감사 기록을 생성했습니다.


## Section 2: 영수증 변조하기

영수증의 핵심 목적은 변조가 명확히 드러나는 것입니다. 이를 증명해 봅시다.

영수증의 문자 한 개를 정확히 수정하고 인증이 실패하는 것을 지켜볼 것입니다.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### 무슨 일이 일어났나요?

`policy_id`를 변경했을 때, 정규화된 바이트가 변경되었습니다. 해당 바이트들의 SHA-256 해시가 변경되었습니다. 원래 해시에 대해 생성된 서명이 더 이상 새 해시와 일치하지 않습니다. 검증 결과가 올바르게 `False`를 반환합니다.

공격자가 개인 키를 가지고 있지 않는 한, 영수증의 어떤 필드를 수정해도 검증을 통과할 수 없습니다. 개인 키가 키 보관소에 있고 공개 키가 게시되어 있는 한, 변조를 숨기는 것은 불가능합니다.

직접 시도해 보세요: 위 셀에서 `tool_name`, `agent_id`, 또는 `timestamp`를 수정하고 다시 실행해 보세요. 모든 변경은 유효하지 않은 영수증을 생성합니다.


## Section 3: 체인 영수증 연결하기

단일 영수증은 하나의 작업을 보호합니다. 대부분의 에이전트는 여러 작업을 수행합니다. 전체 시퀀스를 변조 감지 가능하게 만들기 위해, 각 영수증에 이전 영수증의 해시를 포함하여 이전 영수증과 연결합니다.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

누군가 영수증을 제거하거나 순서를 변경하면 체인은 바로 그 지점에서 끊어집니다. 이후 어떤 영수증의 검증도 실패하는데, 이는 해당 영수증의 `previous_receipt_hash`가 실제 이전 영수증의 해시와 더 이상 일치하지 않기 때문입니다.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

이제 중간 영수증을 변조하여 체인을 끊고 다시 검증하세요. 변조된 영수증은 서명 검증에 실패하며, 다음 영수증은 체인 연결 검증에 실패합니다(변경된 중간 영수증의 해시와 `previous_receipt_hash`가 더 이상 일치하지 않기 때문입니다).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

영수증 0은 여전히 검증됩니다(수정되지 않았고 의존할 이전 영수증이 없기 때문입니다). 영수증 1은 `tool_args_hash`를 변경했기 때문에 서명 검사에서 실패합니다. 영수증 2는 `previous_receipt_hash`가 원본(현재 수정된) 영수증 1을 기준으로 계산되었으므로 체인 연결 검사에서 실패합니다.

공격자가 수정된 영수증 1에 대해 재서명할 수 있다고 하더라도(비밀 키 없이는 할 수 없습니다), 영수증 2에서의 체인 연결 불일치가 변조를 드러내게 됩니다. 변경을 숨기려면 공격자는 수정 시점 이후의 모든 영수증에 대해 재서명을 해야 하며, 이는 비밀 키를 소유하고 있어야 가능합니다.


## Section 4: 에이전트 도구 호출에 영수증 서명 적용하기

실제 배포 환경에서는 모든 에이전트 작성자가 `make_receipt`를 호출하는 것을 기억하게 하고 싶지 않습니다. 모든 도구 호출에 대해 영수증 서명이 자동으로 이루어지길 원합니다.

가장 간단한 패턴은 어떤 호출 가능한 도구 함수도 받아서 영수증을 생성하는 버전으로 반환하는 래퍼 클래스입니다. 이는 Microsoft Agent Framework(`agent_framework.azure`)를 포함한 모든 에이전트 프레임워크에 적용할 수 있습니다.

Azure AI Foundry 프로젝트가 설정되어 있지 않더라도, 아래의 로컬 모의 코드는 이 패턴을 시연합니다.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft Agent Framework와 통합하기

위의 `ReceiptedTool` 래퍼는 프레임워크에 구애받지 않습니다. Microsoft Agent Framework로 구축된 에이전트 내부에서 사용하려면, 래핑된 함수를 도구로 등록하세요. 다음은 스케치이며(모의 코드를 실제 Azure AI Foundry 도구 등록으로 대체할 수 있습니다):

```python
# 통합 형태를 보여주는 의사코드입니다.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     instructions="당신은 Contoso 여행사 에이전트입니다 ...",
#     tools=[receipted_lookup],   # 래핑된 도구, 원시 함수가 아님
# )
# response = agent.run("6월에 시드니에서 로스앤젤레스로 가는 항공편을 찾아주세요.")
#
# # 실행 후, 에이전트가 호출한 모든 도구에는 서명된 영수증이 있습니다:
# audit_chain = receipted_lookup.receipts
```

에이전트 프레임워크는 영수증에 대해 아무것도 알 필요가 없습니다. 영수증 서명은 도구에 감싸여 있으며 프레임워크에 직접 연결되지 않습니다. 이렇게 하면 에이전트를 다시 작성하지 않고도 기존 에이전트 코드에 출처를 추가할 수 있습니다.


## 요약 및 확장 도전 과제

당신은:

- Ed25519 키 쌍을 생성했습니다.
- 에이전트 도구 호출에 대한 영수증을 작성하고 서명했습니다.
- 공개 키만으로 오프라인에서 영수증을 검증했습니다.
- 영수증을 변조하고 검증 실패를 관찰했습니다.
- 세 개의 영수증으로 해시 체인 시퀀스를 작성했습니다.
- 체인 중간을 변조하고 서명 실패와 체인 링크 실패를 모두 관찰했습니다.
- 도구 함수를 자동 영수증 서명으로 래핑했습니다.

**확장 도전 과제.** 영수증 스키마를 `request_id` 필드(분산 추적을 위한 UUID)로 확장하세요. `make_receipt`를 업데이트하여 이를 포함시키고, 영수증이 끝까지 검증되는지 확인하세요. 그런 다음 서명 후 필드를 수정하고 검증 실패를 확인하세요. 이것은 정식 인코딩의 모든 바이트가 서명에 어떻게 기여하는지 내면화하도록 강요합니다.

**중요한 경계.** 영수증은 세 가지 그리고 세 가지만 증명합니다: 귀속(이 키가 이 내용을 서명함), 무결성(서명 이후 내용이 변경되지 않음), 순서(이 영수증이 저 영수증보다 나중임). 영수증은 에이전트의 행동이 올바르다는 것, `policy_id`에 명시된 정책이 실제로 평가되었다는 것, 또는 에이전트가 모든 규칙을 따랐다는 것을 증명하지 않습니다. 영수증은 기반입니다. 거버넌스는 당신이 그 위에 구축하는 시스템입니다.

이 경계를 염두에 두고 수업 README를 다시 읽으세요. 팀이 영수증과 관련해 가장 흔히 하는 실수는 "영수증이 있다"는 것이 "우리는 관리되고 있다"는 의미라고 가정하는 것입니다. 그렇지 않습니다. 영수증은 에이전트 행동을 감audit가능하게 만듭니다. 그것이 올바르다는 것을 만들지는 않습니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
